[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_3_policy_iteration_complete.ipynb)

<div style="text-align:center">
    <h1>
        Policy Iteration
    </h1>
</div>
<br>

<div style="text-align:center">
    <p>
        In this notebook we are going to look at a dynamic programming algorithm called policy iteration. In it, we will iteratively interleave two processes: policy evaluation and policy improvement, until the optimal policy and state values are found.
    </p>
</div>

<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_values


## Import the necessary software libraries:

In [ ]:
# NumPy gives us fast array math for the value table and policy.
import numpy as np
# Matplotlib is used to draw the maze, the policy arrows and the value heatmap.
import matplotlib.pyplot as plt

## Initialize the environment

In [ ]:
# Create the 5x5 grid-world maze. `env` knows the rules: how actions move the
# agent, what reward each step gives, and where the goal is.
env = Maze()

In [ ]:
# Ask the environment for a picture (an RGB image array) of the current maze.
frame = env.render(mode='rgb_array')
# Hide the x/y axis ticks so we only see the maze itself.
plt.axis('off')
# Display the maze image.
plt.imshow(frame)

In [ ]:
# The observation (state) is the agent's grid cell: a (row, col) pair, each 0-4.
print(f"Observation space shape: {env.observation_space.nvec}")
# There are 4 possible actions (up, down, left, right).
print(f"Number of actions: {env.action_space.n}")

## Define the policy $\pi(\cdot|s)$

#### Create the policy $\pi(\cdot|s)$

In [ ]:
# The policy stores, for every state, the probability of choosing each action.
# Shape (5, 5, 4): one row, one column, and 4 action-probabilities per cell.
# We start with 0.25 everywhere, i.e. a uniform random policy (each action equally likely).
policy_probs = np.full((5, 5, 4), 0.25)

In [ ]:
# A small helper: given a state (row, col), return its 4 action probabilities.
def policy(state):
    return policy_probs[state]

#### Test the policy with state (0, 0)

In [ ]:
# Look up the action probabilities the policy assigns at the top-left cell (0, 0).
action_probabilities = policy((0,0))
# Print each action's probability; for the random policy they should all be 0.25.
for action, prob in zip(range(4), action_probabilities):
    print(f"Probability of taking action {action}: {prob}")

#### See how the random policy does in the maze

In [ ]:
# Run the (still random) policy in the maze for one episode and animate it,
# so we can see how badly an untrained agent wanders around.
test_agent(env, policy, episodes=1)

#### Plot the policy

In [ ]:
# Draw the policy as arrows on top of the maze image.
plot_policy(policy_probs, frame)

## Define value table $V(s)$

#### Create the $V(s)$ table

In [ ]:
# The value table V(s): one number per cell estimating how good it is to be there
# (the expected total future reward). We start every cell at 0.
state_values = np.zeros(shape=(5,5))

#### Plot $V(s)$

In [ ]:
# Draw the value table as a heatmap on top of the maze (all zeros for now).
plot_values(state_values, frame)

## Implement the Policy Iteration algorithm

**The big idea (in plain language).**
We want the best plan of action (the *optimal policy*) for every cell in the maze.
Policy iteration finds it by repeating two steps until nothing changes:

1. **Policy evaluation** - *How good is my current plan?*
   Keeping the policy fixed, we compute a value `V(s)` for every state: the expected
   total (discounted) future reward you would collect if you started there and followed
   the current policy. We do this by sweeping all states over and over, each time updating
   a state's value from its neighbours' values, until the numbers settle down (stop changing).

2. **Policy improvement** - *Can I do better?*
   Now we look at those values and, in every state, switch to the action that leads to the
   best-valued outcome. This makes the policy *greedy* with respect to the values.

We then evaluate the new (improved) policy, improve it again, and so on. Each round the policy
is guaranteed to be at least as good as before. When an improvement step changes *nothing*,
the policy is optimal and we stop.

A couple of symbols used below:
- `gamma` (the discount factor, 0.99) makes rewards that arrive sooner worth a little more than
  rewards far in the future.
- `theta` is a tiny threshold: once no value changes by more than this, we treat evaluation as converged.
- `Q(s, a)` (called `qsa` in the code) is the value of taking action `a` in state `s` and then
  acting well afterwards: `reward + gamma * V(next_state)`.

In [ ]:
# POLICY EVALUATION: given a fixed policy, compute how good each state is under it.
# It repeatedly sweeps every state and updates V(s) until the numbers stop changing.
def policy_evaluation(policy_probs, state_values, theta=1e-6, gamma=0.99):
    # `delta` tracks the biggest change to any value in a sweep; start it huge so we enter the loop.
    delta = float("inf")

    # Keep sweeping while some value still changed by more than the tiny threshold theta.
    while delta > theta:
        # Reset the largest-change tracker at the start of each full sweep.
        delta = 0

        # Visit every cell of the 5x5 grid.
        for row in range(5):
            for col in range(5):
                # Remember the current value so we can measure how much it changes.
                old_value = state_values[(row, col)]
                # We will rebuild this state's value from scratch as a weighted average.
                new_value = 0
                # The probabilities this policy assigns to each action in this state.
                action_probabilities = policy_probs[(row, col)]

                # Loop over every action and its probability under the policy.
                for action, prob in enumerate(action_probabilities):
                    # Ask the model what happens if we take this action: where we land and the reward.
                    next_state, reward, _, _ = env.simulate_step((row, col), action)
                    # Bellman expectation update: add this action's probability-weighted contribution
                    # (immediate reward + discounted value of where we land).
                    new_value += prob * (reward + gamma * state_values[next_state])

                # Store the freshly computed value for this state.
                state_values[(row, col)] = new_value

                # Track the largest value change seen this sweep (used to decide when to stop).
                delta = max(delta, abs(old_value - new_value))

In [ ]:
# POLICY IMPROVEMENT: given the values we just computed, make the policy greedy.
# In every state, switch to the action that looks best according to the values.
def policy_improvement(policy_probs, state_values, gamma=0.99):

    # Assume the policy is already optimal; we'll set this False if we change anything.
    policy_stable = True
    # Visit every cell of the grid.
    for row in range(5):
        for col in range(5):
            # The action the current policy prefers here (the one with probability 1).
            old_action = policy_probs[(row, col)].argmax()

            # We'll search for the best action and its value (the action-value Q).
            new_action = None
            max_qsa = float("-inf")

            # Try each of the 4 actions and score it.
            for action in range(4):
                # Simulate the action to see the next state and reward.
                next_state, reward, _, _ = env.simulate_step((row, col), action)
                # Q(s,a): immediate reward plus discounted value of the resulting state.
                qsa = reward + gamma * state_values[next_state]
                # Keep the action with the highest Q value.
                if qsa > max_qsa:
                    max_qsa = qsa
                    new_action = action

            # Make the policy deterministic: probability 1 on the best action, 0 elsewhere.
            action_probs = np.zeros(4)
            action_probs[new_action] = 1.
            policy_probs[(row, col)] = action_probs

            # If the best action differs from the old one, the policy was not yet stable.
            if new_action != old_action:
                policy_stable = False

    # Tell the caller whether the policy stopped changing (True = we're done).
    return policy_stable

In [ ]:
# POLICY ITERATION: alternate evaluation and improvement until the policy stops changing.
def policy_iteration(policy_probs, state_values, theta=1e-6, gamma=0.99):
    # Start by assuming the policy is not yet stable so we enter the loop.
    policy_stable = False

    # Repeat until an improvement step makes no changes (policy is optimal).
    while not policy_stable:

        # Step 1: figure out how good each state is under the current policy.
        policy_evaluation(policy_probs, state_values, theta, gamma)

        # Step 2: make the policy greedy w.r.t. those values; note if anything changed.
        policy_stable = policy_improvement(policy_probs, state_values, gamma)

In [ ]:
# Run the full algorithm. It updates `policy_probs` and `state_values` in place.
policy_iteration(policy_probs, state_values)

## Show results

Show resulting value table V(s)

In [ ]:
# Show the final value heatmap: cells near the goal should now have higher values.
plot_values(state_values, frame)

Show resulting policy $\pi(\cdot|s)$

In [ ]:
# Show the final policy: arrows should now point along the shortest path to the goal.
plot_policy(policy_probs, frame)

#### Test the resulting agent

In [ ]:
# Run the optimal policy in the maze to watch it solve the task efficiently.
test_agent(env, policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 4: Dynamic Programming](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)